## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 15: Projektarbeit & Best Practices
# Niveau: Experten
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import os
import random
import platform
import sys
import time
from datetime import datetime

print("=== Vollständige Reproduzierbarkeit und Wissenschaftlichkeit ===\n")

# ═══════════════════════════════════════════════════════════
# SCHRITT 1: ALLE SEEDS SETZEN
# ═══════════════════════════════════════════════════════════

SEED = 42

def set_all_seeds(seed):
    """
    Setzt alle relevanten Zufalls-Seeds für vollständige Reproduzierbarkeit.

    Betrifft:
    - Python random Modul
    - NumPy
    - TensorFlow / Keras
    - PYTHONHASHSEED (Dictionary-Reihenfolge)
    """
    # Python standard random
    random.seed(seed)
    # NumPy
    np.random.seed(seed)
    # TensorFlow
    tf.random.set_seed(seed)
    # PYTHONHASHSEED (beeinflusst Set/Dict Ordnung)
    os.environ['PYTHONHASHSEED'] = str(seed)

    print(f"Seeds gesetzt: python.random={seed}, numpy={seed}, "
          f"tensorflow={seed}, PYTHONHASHSEED={seed}")

set_all_seeds(SEED)

# ═══════════════════════════════════════════════════════════
# SCHRITT 2: ENVIRONMENT DOKUMENTIEREN
# ═══════════════════════════════════════════════════════════

def get_environment_info():
    """Dokumentiert die komplette Software-Umgebung."""
    env = {
        'timestamp':     datetime.now().isoformat(),
        'python_version': sys.version,
        'platform':      platform.platform(),
        'processor':     platform.processor(),
        'tensorflow_version': tf.__version__,
        'keras_version': keras.__version__,
        'numpy_version': np.__version__,
        'seed_used':     SEED,
        'gpu_available': len(tf.config.list_physical_devices('GPU')) > 0,
        'n_gpus':        len(tf.config.list_physical_devices('GPU')),
    }

    # Weitere Package-Versionen
    try:
        import sklearn; env['sklearn_version'] = sklearn.__version__
    except ImportError:
        env['sklearn_version'] = 'nicht installiert'

    try:
        import matplotlib; env['matplotlib_version'] = matplotlib.__version__
    except ImportError:
        env['matplotlib_version'] = 'nicht installiert'

    return env

env_info = get_environment_info()
print("\n=== UMGEBUNGS-DOKUMENTATION ===")
for k, v in env_info.items():
    print(f"  {k}: {v}")

# ═══════════════════════════════════════════════════════════
# SCHRITT 3: REPRODUZIERBARES EXPERIMENT
# ═══════════════════════════════════════════════════════════

def run_experiment(seed=SEED):
    """
    Führt ein vollständig reproduzierbares Experiment durch.
    Bei gleichem Seed und gleicher Hardware identische Ergebnisse.
    """
    set_all_seeds(seed)

    # Daten laden
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    x_train = x_train.astype('float32') / 255.0
    x_test  = x_test.astype('float32') / 255.0
    x_train = x_train.reshape(-1, 784)
    x_test  = x_test.reshape(-1, 784)

    # Kleines Subset für schnelle Verifikation
    x_train_sub = x_train[:2000]
    y_train_sub = y_train[:2000]

    # Modell definieren
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(784,),
                     kernel_initializer=keras.initializers.GlorotUniform(seed=seed)),
        layers.Dense(32, activation='relu',
                     kernel_initializer=keras.initializers.GlorotUniform(seed=seed)),
        layers.Dense(10, activation='softmax',
                     kernel_initializer=keras.initializers.GlorotUniform(seed=seed))
    ], name='reproducible_model')

    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Training
    hist = model.fit(x_train_sub, y_train_sub,
                      epochs=5, batch_size=64,
                      validation_split=0.2, verbose=0)

    # Vorhersagen auf festen Testsamples
    preds = model.predict(x_test[:10], verbose=0)
    _, test_acc = model.evaluate(x_test, y_test, verbose=0)

    return {
        'final_val_acc': hist.history['val_accuracy'][-1],
        'test_acc': test_acc,
        'predictions_10': preds.tolist(),
        'first_pred': int(np.argmax(preds[0])),
        'val_acc_history': hist.history['val_accuracy']
    }

# ═══════════════════════════════════════════════════════════
# SCHRITT 4: REPRODUZIERBARKEIT VERIFIZIEREN (2× ausführen)
# ═══════════════════════════════════════════════════════════

print("\n=== REPRODUZIERBARKEITS-TEST ===")
print("Führe identisches Experiment zweimal durch...\n")

t1_start = time.time()
result_1 = run_experiment(seed=SEED)
t1 = time.time() - t1_start

t2_start = time.time()
result_2 = run_experiment(seed=SEED)
t2 = time.time() - t2_start

# Vergleichen
print(f"Lauf 1: Val-Acc={result_1['final_val_acc']:.6f}, "
      f"Test-Acc={result_1['test_acc']:.6f}, Zeit={t1:.2f}s")
print(f"Lauf 2: Val-Acc={result_2['final_val_acc']:.6f}, "
      f"Test-Acc={result_2['test_acc']:.6f}, Zeit={t2:.2f}s")

# Prüfe numerische Identität
preds1 = np.array(result_1['predictions_10'])
preds2 = np.array(result_2['predictions_10'])
numerically_identical = np.allclose(preds1, preds2, atol=1e-6)
val_acc_identical = abs(result_1['final_val_acc'] - result_2['final_val_acc']) < 1e-6

print(f"\nNumerisch identische Vorhersagen: {numerically_identical}")
print(f"Identische Val-Acc:              {val_acc_identical}")

if numerically_identical and val_acc_identical:
    print("✓ VOLLSTÄNDIG REPRODUZIERBAR!")
else:
    print("⚠ Kleine numerische Unterschiede (GPU-Parallelismus kann nicht-deterministisch sein)")
    print("  Tipp: TF_DETERMINISTIC_OPS=1 setzen für vollständigen Determinismus auf GPU")

# ═══════════════════════════════════════════════════════════
# SCHRITT 5: REPRODUZIERBARKEITS-REPORT SPEICHERN
# ═══════════════════════════════════════════════════════════

report = {
    'timestamp':   datetime.now().isoformat(),
    'environment': env_info,
    'seed':        SEED,
    'experiment': {
        'description': 'MNIST Klassifikation (2000 Samples, 5 Epochen)',
        'model': '784→64→32→10',
        'initializer': f'GlorotUniform(seed={SEED})'
    },
    'reproducibility_test': {
        'run_1': {
            'final_val_acc': result_1['final_val_acc'],
            'test_acc': result_1['test_acc'],
            'first_prediction': result_1['first_pred']
        },
        'run_2': {
            'final_val_acc': result_2['final_val_acc'],
            'test_acc': result_2['test_acc'],
            'first_prediction': result_2['first_pred']
        },
        'numerically_identical': bool(numerically_identical),
        'val_acc_identical': bool(val_acc_identical),
        'verdict': 'REPRODUZIERBAR' if numerically_identical else 'KLEINE UNTERSCHIEDE'
    }
}

report_path = 'reproducibility_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"\nReport gespeichert: {report_path}")

# ═══════════════════════════════════════════════════════════
# VISUALISIERUNG
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Reproduzierbarkeit: Lauf 1 vs. Lauf 2', fontsize=13)

# Lernkurven Vergleich
ax = axes[0]
ax.plot(result_1['val_acc_history'], label='Lauf 1', color='blue', linewidth=2)
ax.plot(result_2['val_acc_history'], label='Lauf 2', color='red', linestyle='--', linewidth=2)
ax.set_title('Val-Accuracy: Lauf 1 vs. Lauf 2')
ax.set_xlabel('Epoche'); ax.set_ylabel('Val-Accuracy')
ax.legend(); ax.grid(True)

# Vorhersagen Vergleich (10 Samples)
ax2 = axes[1]
im = ax2.imshow(np.abs(preds1 - preds2), cmap='Reds', aspect='auto')
ax2.set_title('|Preds1 - Preds2| (Abweichung)')
ax2.set_xlabel('Klasse (0-9)'); ax2.set_ylabel('Sample')
plt.colorbar(im, ax=ax2)

# Zusammenfassung
ax3 = axes[2]
ax3.axis('off')
max_diff = np.abs(preds1 - preds2).max()
mean_diff = np.abs(preds1 - preds2).mean()
summary = f"""Reproduzierbarkeitsbericht:

Seed: {SEED}
TF-Version: {tf.__version__}
GPU: {env_info['gpu_available']}

Lauf 1:
  Val-Acc:  {result_1['final_val_acc']:.6f}
  Test-Acc: {result_1['test_acc']:.6f}

Lauf 2:
  Val-Acc:  {result_2['final_val_acc']:.6f}
  Test-Acc: {result_2['test_acc']:.6f}

Abweichung:
  Max |Δ pred|:  {max_diff:.2e}
  Mean |Δ pred|: {mean_diff:.2e}

Ergebnis: {report['reproducibility_test']['verdict']}"""
ax3.text(0.05, 0.95, summary, transform=ax3.transAxes, fontsize=9,
          va='top', fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='lightgreen' if numerically_identical else 'lightyellow'))

plt.tight_layout()
plt.savefig('E15_1_reproduzierbarkeit.png', dpi=100)
plt.show()

print("\nReprozierbarkeitsstudie abgeschlossen!")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 15: Projektarbeit & Best Practices
# Niveau: Experten
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

"""
Production-Ready Code: MNIST Klassifikation

Transformiert einfaches Modellskript in produktionsreifen Code:
- Typhinweise (Type Hints)
- Dataclass-Konfiguration
- Vollständige Dokumentation (Docstrings)
- Fehlerbehandlung (try/except)
- Python-Logging-Modul
- Input-Validierung
- Unit-Tests (unittest)
- Performance-Benchmarking
"""

# ── Standard-Importe ──────────────────────────────────────
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import logging
import time
import unittest
from dataclasses import dataclass, field
from typing import Tuple, Optional, Dict, List, Any

# ── Logging einrichten ────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('mnist_production')

# ── Konfiguration als Dataclass ───────────────────────────
@dataclass
class ModelConfig:
    """
    Konfigurationsklasse für das MNIST-Modell.

    Alle Hyperparameter werden hier zentral definiert.
    Kein 'Magic Number' im Code!
    """
    hidden_units: Tuple[int, ...] = (128, 64)
    activation:   str             = 'relu'
    dropout_rate: float           = 0.2
    learning_rate: float          = 0.001
    batch_size:   int             = 256
    epochs:       int             = 10
    patience:     int             = 5
    seed:         int             = 42
    model_name:   str             = 'production_mnist'

    def __post_init__(self) -> None:
        """Validiert die Konfiguration nach der Initialisierung."""
        if not (0 < self.learning_rate < 1):
            raise ValueError(f"Learning rate muss zwischen 0 und 1 sein, ist: {self.learning_rate}")
        if not (0 <= self.dropout_rate < 1):
            raise ValueError(f"Dropout muss zwischen 0 und 1 sein, ist: {self.dropout_rate}")
        if self.batch_size <= 0:
            raise ValueError(f"Batch-Size muss positiv sein, ist: {self.batch_size}")


@dataclass
class DataConfig:
    """Konfiguration für Datenladen und -verarbeitung."""
    dataset:       str   = 'mnist'
    val_fraction:  float = 0.1
    n_train_limit: Optional[int] = None   # None = alle


# ── Daten-Klasse ──────────────────────────────────────────
class MNISTDataLoader:
    """
    Lädt und verarbeitet MNIST-Daten.

    Attributes:
        config: DataConfig Objekt mit Konfiguration
    """

    def __init__(self, config: DataConfig) -> None:
        self.config = config
        self._logger = logging.getLogger(f'{__name__}.DataLoader')

    def load(self) -> Dict[str, np.ndarray]:
        """
        Lädt und preprocesst MNIST-Daten.

        Returns:
            Dict mit x_train, y_train, x_val, y_val, x_test, y_test

        Raises:
            RuntimeError: Wenn Daten nicht geladen werden können
        """
        try:
            self._logger.info("Lade MNIST-Daten...")
            (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

            # Normalisieren
            x_train = x_train.astype('float32') / 255.0
            x_test  = x_test.astype('float32') / 255.0

            # Flatten
            x_train = x_train.reshape(-1, 784)
            x_test  = x_test.reshape(-1, 784)

            # Optionales Limitieren
            if self.config.n_train_limit:
                x_train = x_train[:self.config.n_train_limit]
                y_train = y_train[:self.config.n_train_limit]
                self._logger.info(f"Training auf {self.config.n_train_limit} Samples begrenzt")

            # Validierungssplit
            n_val    = int(len(x_train) * self.config.val_fraction)
            x_val    = x_train[-n_val:]
            y_val    = y_train[-n_val:]
            x_train  = x_train[:-n_val]
            y_train  = y_train[:-n_val]

            data = {
                'x_train': x_train, 'y_train': y_train,
                'x_val':   x_val,   'y_val':   y_val,
                'x_test':  x_test,  'y_test':  y_test
            }

            self._logger.info(
                f"Daten geladen: Train={x_train.shape[0]}, "
                f"Val={x_val.shape[0]}, Test={x_test.shape[0]}"
            )
            return data

        except Exception as e:
            self._logger.error(f"Fehler beim Laden der Daten: {e}")
            raise RuntimeError(f"Datenladen fehlgeschlagen: {e}") from e

    def validate_input(self, x: np.ndarray) -> None:
        """
        Validiert Eingabedaten für Inferenz.

        Args:
            x: Eingabe-Array

        Raises:
            ValueError: Wenn Eingabe ungültig
            TypeError:  Wenn falscher Datentyp
        """
        if not isinstance(x, np.ndarray):
            raise TypeError(f"Eingabe muss np.ndarray sein, ist: {type(x)}")
        if x.ndim == 1:
            x = x.reshape(1, -1)
        if x.shape[-1] != 784:
            raise ValueError(f"Eingabe muss 784 Features haben, hat: {x.shape[-1]}")
        if x.dtype != np.float32:
            raise ValueError(f"Eingabe muss float32 sein, ist: {x.dtype}")
        if np.any(x < 0) or np.any(x > 1):
            raise ValueError("Pixelwerte müssen in [0, 1] liegen")


# ── Modell-Klasse ─────────────────────────────────────────
class MNISTClassifier:
    """
    Produktionsreifer MNIST-Klassifikator.

    Attributes:
        config:  ModelConfig Objekt
        model:   Keras Sequential Model (nach build())
        history: Training History (nach train())
    """

    def __init__(self, config: ModelConfig) -> None:
        self.config  = config
        self.model:  Optional[keras.Model] = None
        self.history: Optional[Any] = None
        self._logger = logging.getLogger(f'{__name__}.Classifier')

        # Seeds setzen
        np.random.seed(config.seed)
        tf.random.set_seed(config.seed)

    def build(self, input_dim: int = 784, n_classes: int = 10) -> keras.Model:
        """
        Erstellt das Modell.

        Args:
            input_dim: Eingabe-Dimension
            n_classes: Anzahl der Klassen

        Returns:
            Kompiliertes Keras Modell

        Raises:
            ValueError: Wenn Konfiguration ungültig
        """
        try:
            model_layers: List[layers.Layer] = []

            for i, units in enumerate(self.config.hidden_units):
                if i == 0:
                    model_layers.append(
                        layers.Dense(units, activation=self.config.activation,
                                      input_shape=(input_dim,))
                    )
                else:
                    model_layers.append(
                        layers.Dense(units, activation=self.config.activation)
                    )
                if self.config.dropout_rate > 0:
                    model_layers.append(layers.Dropout(self.config.dropout_rate))

            model_layers.append(layers.Dense(n_classes, activation='softmax'))

            self.model = keras.Sequential(model_layers, name=self.config.model_name)
            self.model.compile(
                optimizer=keras.optimizers.Adam(self.config.learning_rate),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )

            self._logger.info(f"Modell erstellt: {self.model.count_params():,} Parameter")
            return self.model

        except Exception as e:
            self._logger.error(f"Fehler beim Erstellen des Modells: {e}")
            raise

    def train(self, data: Dict[str, np.ndarray]) -> Any:
        """
        Trainiert das Modell.

        Args:
            data: Dict mit x_train, y_train, x_val, y_val

        Returns:
            Keras History Objekt

        Raises:
            RuntimeError: Wenn Modell nicht gebaut wurde
        """
        if self.model is None:
            raise RuntimeError("Modell nicht gebaut. build() zuerst aufrufen.")

        self._logger.info(f"Starte Training (max {self.config.epochs} Epochen)...")
        t0 = time.time()

        try:
            callbacks = [
                keras.callbacks.EarlyStopping(
                    monitor='val_loss',
                    patience=self.config.patience,
                    restore_best_weights=True
                )
            ]

            self.history = self.model.fit(
                data['x_train'], data['y_train'],
                validation_data=(data['x_val'], data['y_val']),
                epochs=self.config.epochs,
                batch_size=self.config.batch_size,
                callbacks=callbacks,
                verbose=0
            )

            elapsed = time.time() - t0
            n_epochs = len(self.history.history['loss'])
            best_val = max(self.history.history['val_accuracy'])
            self._logger.info(
                f"Training abgeschlossen: {n_epochs} Epochen, "
                f"beste Val-Acc={best_val:.4f}, Zeit={elapsed:.1f}s"
            )
            return self.history

        except Exception as e:
            self._logger.error(f"Fehler beim Training: {e}")
            raise

    def predict(self, x: np.ndarray) -> np.ndarray:
        """
        Führt Vorhersagen durch.

        Args:
            x: Eingabe-Array (n_samples, 784) oder (784,)

        Returns:
            Vorhersage-Wahrscheinlichkeiten (n_samples, n_classes)

        Raises:
            RuntimeError: Wenn Modell nicht trainiert
            ValueError:   Bei ungültiger Eingabe
        """
        if self.model is None:
            raise RuntimeError("Modell nicht trainiert.")

        if x.ndim == 1:
            x = x.reshape(1, -1)

        return self.model.predict(x, verbose=0)

    def evaluate(self, x_test: np.ndarray, y_test: np.ndarray) -> Dict[str, float]:
        """
        Evaluiert auf Test-Daten.

        Returns:
            Dict mit loss und accuracy
        """
        if self.model is None:
            raise RuntimeError("Modell nicht trainiert.")

        loss, acc = self.model.evaluate(x_test, y_test, verbose=0)
        self._logger.info(f"Test-Evaluation: Loss={loss:.4f}, Acc={acc:.4f}")
        return {'loss': float(loss), 'accuracy': float(acc)}

    def benchmark(self, x: np.ndarray, n_runs: int = 100) -> Dict[str, float]:
        """
        Benchmark der Inferenz-Geschwindigkeit.

        Args:
            x: Eingabe-Array
            n_runs: Anzahl der Durchläufe

        Returns:
            Dict mit mean, std, p95 Latenz in ms
        """
        latencies = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            self.predict(x)
            latencies.append((time.perf_counter() - t0) * 1000)

        return {
            'mean_ms': float(np.mean(latencies)),
            'std_ms':  float(np.std(latencies)),
            'p95_ms':  float(np.percentile(latencies, 95)),
            'n_runs':  n_runs
        }


# ── Unit Tests ────────────────────────────────────────────
class TestMNISTClassifier(unittest.TestCase):
    """Unit-Tests für MNISTClassifier."""

    @classmethod
    def setUpClass(cls):
        """Setzt einmalig Testfixtures auf."""
        cls.config   = ModelConfig(hidden_units=(32,), epochs=2, batch_size=256)
        cls.clf      = MNISTClassifier(cls.config)
        cls.clf.build()

        # Minimale Testdaten
        cls.data = {
            'x_train': np.random.rand(200, 784).astype('float32'),
            'y_train': np.random.randint(0, 10, 200).astype('int32'),
            'x_val':   np.random.rand(50, 784).astype('float32'),
            'y_val':   np.random.randint(0, 10, 50).astype('int32'),
        }
        cls.clf.train(cls.data)

    def test_model_builds(self):
        """Testet, dass Modell korrekt aufgebaut wird."""
        self.assertIsNotNone(self.clf.model)
        self.assertGreater(self.clf.model.count_params(), 0)

    def test_predict_shape(self):
        """Testet, dass Vorhersagen die richtige Form haben."""
        x = np.random.rand(5, 784).astype('float32')
        preds = self.clf.predict(x)
        self.assertEqual(preds.shape, (5, 10))

    def test_predict_probabilities_sum_to_one(self):
        """Testet, dass Wahrscheinlichkeiten auf 1 summieren."""
        x = np.random.rand(10, 784).astype('float32')
        preds = self.clf.predict(x)
        self.assertTrue(np.allclose(preds.sum(axis=1), 1.0, atol=1e-5))

    def test_config_validation(self):
        """Testet, dass ungültige Konfigurationen erkannt werden."""
        with self.assertRaises(ValueError):
            ModelConfig(learning_rate=2.0)   # LR > 1 ungültig
        with self.assertRaises(ValueError):
            ModelConfig(dropout_rate=-0.1)   # Negativer Dropout

    def test_single_sample_prediction(self):
        """Testet Vorhersage für ein einzelnes Sample."""
        x = np.random.rand(784).astype('float32')
        preds = self.clf.predict(x)
        self.assertEqual(preds.shape, (1, 10))


# ── HAUPTPROGRAMM ─────────────────────────────────────────
if __name__ == '__main__':
    logger.info("Starte Production-Ready MNIST Klassifikation")

    # Konfiguration
    model_cfg = ModelConfig(
        hidden_units=(128, 64),
        dropout_rate=0.2,
        learning_rate=0.001,
        epochs=15,
        batch_size=256
    )
    data_cfg = DataConfig(n_train_limit=5000)

    logger.info(f"Konfiguration: {model_cfg}")

    # Daten laden
    loader = MNISTDataLoader(data_cfg)
    data   = loader.load()

    # Modell
    clf = MNISTClassifier(model_cfg)
    clf.build()

    # Training
    history = clf.train(data)

    # Evaluation
    results = clf.evaluate(data['x_test'], data['y_test'])
    logger.info(f"Finale Ergebnisse: {results}")

    # Benchmarking
    logger.info("Starte Performance-Benchmark (100 Runs, 1 Sample)...")
    bench = clf.benchmark(data['x_test'][:1], n_runs=100)
    logger.info(f"Benchmark: mean={bench['mean_ms']:.2f}ms, "
                f"std={bench['std_ms']:.2f}ms, p95={bench['p95_ms']:.2f}ms")

    # Unit Tests ausführen
    print("\n=== Unit-Tests ===")
    suite = unittest.TestLoader().loadTestsFromTestCase(TestMNISTClassifier)
    runner = unittest.TextTestRunner(verbosity=2)
    test_result = runner.run(suite)

    # Visualisierung
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Production-Ready MNIST Klassifikation', fontsize=13)

    ax = axes[0]
    ax.plot(history.history['accuracy'],     label='Train')
    ax.plot(history.history['val_accuracy'], label='Val')
    ax.set_title('Lernkurven'); ax.set_xlabel('Epoche')
    ax.set_ylabel('Accuracy'); ax.legend(); ax.grid(True)

    ax2 = axes[1]
    benchmark_data = [bench['mean_ms'], bench['std_ms'], bench['p95_ms']]
    ax2.bar(['Mean', 'Std', 'P95'], benchmark_data, color=['blue', 'orange', 'red'], alpha=0.8)
    ax2.set_title('Inferenz-Benchmark (ms/Sample)')
    ax2.set_ylabel('Millisekunden'); ax2.grid(True, axis='y')

    ax3 = axes[2]
    ax3.axis('off')
    summary = f"""Production Code Qualitätsmerkmale:

✓ Type Hints
✓ Docstrings (Einzel-/Klassen-Methoden)
✓ Dataclass-Konfiguration
✓ Python Logging (statt print)
✓ Fehlerbehandlung (try/except)
✓ Input-Validierung
✓ Unit-Tests (unittest)
✓ Performance-Benchmarking

Ergebnisse:
  Test-Acc: {results['accuracy']:.4f}
  Mean Latenz: {bench['mean_ms']:.2f}ms
  Tests: {test_result.testsRun} laufen, 
         {len(test_result.errors)+len(test_result.failures)} Fehler"""
    ax3.text(0.05, 0.95, summary, transform=ax3.transAxes, fontsize=9,
              va='top', fontfamily='monospace',
              bbox=dict(boxstyle='round', facecolor='lightgreen'))

    plt.tight_layout()
    plt.savefig('E15_2_production_ready_code.png', dpi=100)
    plt.show()

    logger.info("Production-Ready Code Demonstration abgeschlossen.")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 15: Projektarbeit
# Niveau: Experten
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

"""
Abschlussprojekt: Vollständiges KI-System
==========================================
Dieses Skript integriert alle Kernkomponenten eines produktionsreifen
KI-Systems in einer modularen, gut dokumentierten Architektur:

  1. DataPipeline         – Laden, Normalisieren, Augmentieren, Caching
  2. CNNFeatureExtractor  – Konvolutionaler Feature-Extraktor
  3. LSTMTemporalModel    – Zeitreihen-Modell mit LSTM
  4. EnsemblePredictor    – Gewichtetes Ensemble mehrerer Modelle
  5. UncertaintyEstimator – MC-Dropout-basierte Unsicherheitsschätzung
  6. DeploymentEndpoint   – Simulierter REST-Endpunkt (Flask-ähnlich)
  7. ModelMonitor         – Drift-Erkennung und Performance-Tracking
  8. ABTestFramework      – A/B-Test-Experiment für zwei Modellversionen

Alle Module kommunizieren über klar definierte Schnittstellen und
können unabhängig getestet werden.
"""

import os
import time
import json
import hashlib
import logging
import datetime
import numpy as np
import tensorflow as tf
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Any
import matplotlib

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Reproduzierbarkeit ──────────────────────────────────────
np.random.seed(42)
tf.random.set_seed(42)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ── Logging konfigurieren ───────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-25s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('KI_System')

# ============================================================
# 1. DATENPIPELINE
# ============================================================

@dataclass
class DataConfig:
    """Konfiguration der Datenpipeline."""
    batch_size: int = 64
    val_split: float = 0.15
    test_split: float = 0.10
    augment: bool = True
    normalize: bool = True
    cache: bool = True
    num_classes: int = 10
    sequence_length: int = 16   # Für das LSTM-Modell
    image_size: Tuple[int, int] = (28, 28)


class DataPipeline:
    """
    Vollständige Datenpipeline mit Augmentierung und Caching.

    Methoden:
        load()        – Rohdaten laden und aufteilen
        build()       – tf.data.Dataset-Objekte erstellen
        get_splits()  – (train_ds, val_ds, test_ds) zurückgeben
    """

    def __init__(self, config: DataConfig):
        self.config = config
        self.logger = logging.getLogger('DataPipeline')
        self._built = False

        # Interne Speicher
        self._x_train: Optional[np.ndarray] = None
        self._y_train: Optional[np.ndarray] = None
        self._x_val:   Optional[np.ndarray] = None
        self._y_val:   Optional[np.ndarray] = None
        self._x_test:  Optional[np.ndarray] = None
        self._y_test:  Optional[np.ndarray] = None

    # ── Laden ─────────────────────────────────────────────
    def load(self) -> 'DataPipeline':
        """MNIST laden und in Train/Val/Test aufteilen."""
        self.logger.info("Lade MNIST-Datensatz …")
        (x_tr, y_tr), (x_te, y_te) = tf.keras.datasets.mnist.load_data()

        # Normalisieren
        if self.config.normalize:
            x_tr = x_tr.astype('float32') / 255.0
            x_te = x_te.astype('float32') / 255.0

        # Kanal-Dimension ergänzen → (N, 28, 28, 1)
        x_tr = x_tr[..., np.newaxis]
        x_te = x_te[..., np.newaxis]

        # Validierungsset aus Trainingsdaten abschneiden
        n_val = int(len(x_tr) * self.config.val_split)
        idx = np.random.permutation(len(x_tr))
        val_idx, train_idx = idx[:n_val], idx[n_val:]

        self._x_train, self._y_train = x_tr[train_idx], y_tr[train_idx]
        self._x_val,   self._y_val   = x_tr[val_idx],   y_tr[val_idx]
        self._x_test,  self._y_test  = x_te,             y_te

        self.logger.info(
            f"Train: {len(self._x_train):,}  Val: {len(self._x_val):,}  "
            f"Test: {len(self._x_test):,}"
        )
        return self

    # ── Augmentierung ──────────────────────────────────────
    @staticmethod
    def _augment(image: tf.Tensor, label: tf.Tensor):
        """Einfache Augmentierungsschritte für Bilder."""
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, max_delta=0.1)
        image = tf.clip_by_value(image, 0.0, 1.0)
        return image, label

    # ── Aufbauen ───────────────────────────────────────────
    def build(self) -> 'DataPipeline':
        """tf.data.Dataset-Objekte aus den Arrays erstellen."""
        cfg = self.config

        def make_ds(x, y, augment=False):
            ds = tf.data.Dataset.from_tensor_slices((x, y))
            if augment and cfg.augment:
                ds = ds.map(self._augment, num_parallel_calls=tf.data.AUTOTUNE)
            ds = ds.shuffle(1024).batch(cfg.batch_size)
            if cfg.cache:
                ds = ds.cache()
            return ds.prefetch(tf.data.AUTOTUNE)

        self._train_ds = make_ds(self._x_train, self._y_train, augment=True)
        self._val_ds   = make_ds(self._x_val,   self._y_val)
        self._test_ds  = make_ds(self._x_test,  self._y_test)
        self._built = True
        self.logger.info("Datasets aufgebaut und gecached.")
        return self

    # ── Zugriff ────────────────────────────────────────────
    def get_splits(self):
        if not self._built:
            raise RuntimeError("Zuerst build() aufrufen.")
        return self._train_ds, self._val_ds, self._test_ds

    @property
    def raw_test(self):
        return self._x_test, self._y_test


# ============================================================
# 2. CNN FEATURE EXTRACTOR
# ============================================================

class CNNFeatureExtractor(tf.keras.Model):
    """
    Konvolutionaler Feature-Extraktor.

    Erzeugt einen 128-dimensionalen Embedding-Vektor pro Bild.
    Kann als Backbone für andere Modelle verwendet werden.
    """

    def __init__(self, embedding_dim: int = 128, dropout_rate: float = 0.3,
                 name: str = 'CNN_FeatureExtractor'):
        super().__init__(name=name)
        self.embedding_dim = embedding_dim

        # Konvolutionale Blöcke
        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')
        self.bn1   = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(2)

        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')
        self.bn2   = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(2)

        self.conv3 = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')
        self.bn3   = tf.keras.layers.BatchNormalization()
        self.gap   = tf.keras.layers.GlobalAveragePooling2D()

        # Embedding-Schicht
        self.dense_emb = tf.keras.layers.Dense(embedding_dim, activation='relu')
        self.dropout   = tf.keras.layers.Dropout(dropout_rate)
        self.bn_emb    = tf.keras.layers.BatchNormalization()

        # Klassifikationskopf
        self.classifier = tf.keras.layers.Dense(10, activation='softmax',
                                                 name='cnn_output')

    def extract_features(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:
        """Gibt den Embedding-Vektor zurück (kein Klassifikationskopf)."""
        x = self.bn1(self.conv1(x), training=training)
        x = self.pool1(x)
        x = self.bn2(self.conv2(x), training=training)
        x = self.pool2(x)
        x = self.bn3(self.conv3(x), training=training)
        x = self.gap(x)
        x = self.dropout(self.dense_emb(x), training=training)
        return self.bn_emb(x, training=training)

    def call(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:
        features = self.extract_features(x, training=training)
        return self.classifier(features)

    def get_config(self):
        return {'embedding_dim': self.embedding_dim}


# ============================================================
# 3. LSTM TEMPORAL MODEL
# ============================================================

class LSTMTemporalModel(tf.keras.Model):
    """
    LSTM-Modell für die Klassifikation von Sequenzen.

    Eingabe: Sequenz von Feature-Vektoren (batch, seq_len, feature_dim)
    Ausgabe: Klassenwahrscheinlichkeiten (batch, num_classes)
    """

    def __init__(self, feature_dim: int = 128, lstm_units: int = 128,
                 num_classes: int = 10, dropout_rate: float = 0.3,
                 name: str = 'LSTM_TemporalModel'):
        super().__init__(name=name)
        self.feature_dim = feature_dim

        # LSTM-Schichten (bidirektional für bessere Kontexterfassung)
        self.bilstm1 = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(lstm_units, return_sequences=True,
                                  dropout=dropout_rate))
        self.bilstm2 = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(lstm_units // 2,
                                  dropout=dropout_rate))

        # Ausgabeschichten
        self.dropout  = tf.keras.layers.Dropout(dropout_rate)
        self.dense1   = tf.keras.layers.Dense(64, activation='relu')
        self.bn       = tf.keras.layers.BatchNormalization()
        self.output_l = tf.keras.layers.Dense(num_classes, activation='softmax',
                                               name='lstm_output')

    def call(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:
        # x shape: (batch, seq_len, feature_dim)
        x = self.bilstm1(x, training=training)
        x = self.bilstm2(x, training=training)
        x = self.dropout(x, training=training)
        x = self.bn(self.dense1(x), training=training)
        return self.output_l(x)

    def get_config(self):
        return {'feature_dim': self.feature_dim}


# ============================================================
# 4. ENSEMBLE PREDICTOR
# ============================================================

class EnsemblePredictor:
    """
    Gewichtetes Ensemble mehrerer Keras-Modelle.

    Unterstützt:
      - Gleichgewichtetes Averaging (uniform)
      - Performance-gewichtetes Averaging
      - Majority Voting (hard voting)
    """

    def __init__(self, models: List[tf.keras.Model],
                 weights: Optional[List[float]] = None):
        self.models  = models
        self.weights = weights or [1.0 / len(models)] * len(models)
        self.logger  = logging.getLogger('EnsemblePredictor')

        # Gewichte normieren
        total = sum(self.weights)
        self.weights = [w / total for w in self.weights]
        self.logger.info(
            f"Ensemble mit {len(models)} Modellen. "
            f"Gewichte: {[f'{w:.3f}' for w in self.weights]}"
        )

    def predict_proba(self, x: np.ndarray) -> np.ndarray:
        """Gibt gemittelte Wahrscheinlichkeiten zurück."""
        probs = np.zeros((len(x), 10), dtype=np.float32)
        for model, w in zip(self.models, self.weights):
            probs += w * model.predict(x, verbose=0)
        return probs

    def predict(self, x: np.ndarray) -> np.ndarray:
        """Gibt die Klasse mit der höchsten Wahrscheinlichkeit zurück."""
        return np.argmax(self.predict_proba(x), axis=1)

    def evaluate(self, x: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        """Berechnet Accuracy, Top-3-Accuracy und mittlere Konfidenz."""
        probs   = self.predict_proba(x)
        preds   = np.argmax(probs, axis=1)
        acc     = np.mean(preds == y)

        # Top-3 Accuracy
        top3    = np.argsort(probs, axis=1)[:, -3:]
        top3_ok = np.mean([y[i] in top3[i] for i in range(len(y))])

        # Mittlere Konfidenz der Vorhersage
        conf    = np.mean(np.max(probs, axis=1))

        metrics = {'accuracy': float(acc),
                   'top3_accuracy': float(top3_ok),
                   'mean_confidence': float(conf)}
        self.logger.info(f"Ensemble-Evaluation: {metrics}")
        return metrics

    @classmethod
    def from_accuracies(cls, models: List[tf.keras.Model],
                        accuracies: List[float]) -> 'EnsemblePredictor':
        """Erstellt ein Ensemble mit accuracy-proportionalen Gewichten."""
        return cls(models, weights=accuracies)


# ============================================================
# 5. UNCERTAINTY ESTIMATOR (MC-DROPOUT)
# ============================================================

class UncertaintyEstimator:
    """
    Schätzt Vorhersageunsicherheit via Monte-Carlo-Dropout.

    Mehrfache Vorwärtsdurchläufe mit aktiviertem Dropout liefern
    eine Verteilung über Vorhersagen, aus der Epistemic Uncertainty
    und Prädiktive Entropie berechnet werden.
    """

    def __init__(self, model: tf.keras.Model, n_samples: int = 30):
        self.model     = model
        self.n_samples = n_samples
        self.logger    = logging.getLogger('UncertaintyEstimator')

    def predict_with_uncertainty(self, x: np.ndarray) \
            -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Gibt zurück:
          mean_probs    – mittlere Vorhersagewahrscheinlichkeiten (N, C)
          epistemic_unc – Epistemic Uncertainty (Varianz über Samples)  (N,)
          pred_entropy  – Prädiktive Entropie H[p(y|x)]                 (N,)
        """
        # Mehrfache Stichproben mit training=True (Dropout aktiv)
        samples = np.stack(
            [self.model(x, training=True).numpy() for _ in range(self.n_samples)],
            axis=0
        )  # shape: (n_samples, N, C)

        mean_probs    = samples.mean(axis=0)                  # (N, C)
        # Epistemic Uncertainty ≈ Varianz der Vorhersagen
        epistemic_unc = samples.var(axis=0).mean(axis=1)      # (N,)
        # Prädiktive Entropie H = -Σ p·log(p)
        eps           = 1e-8
        pred_entropy  = -(mean_probs * np.log(mean_probs + eps)).sum(axis=1)

        return mean_probs, epistemic_unc, pred_entropy

    def get_confident_predictions(self, x: np.ndarray,
                                  threshold: float = 0.05) \
            -> Tuple[np.ndarray, np.ndarray]:
        """
        Gibt nur Vorhersagen mit geringer Unsicherheit zurück.
        Rückgabe: (Indizes zuverlässiger Samples, Vorhersagen)
        """
        mean_probs, unc, _ = self.predict_with_uncertainty(x)
        reliable_mask      = unc < threshold
        preds              = np.argmax(mean_probs, axis=1)
        self.logger.info(
            f"Zuverlässige Samples: {reliable_mask.sum()}/{len(x)} "
            f"(Schwelle unc < {threshold})"
        )
        return np.where(reliable_mask)[0], preds[reliable_mask]


# ============================================================
# 6. DEPLOYMENT ENDPOINT SIMULATION
# ============================================================

@dataclass
class PredictionRequest:
    """Repräsentiert eine eingehende Anfrage an den Endpunkt."""
    request_id: str
    payload: np.ndarray          # Bild-Tensor
    timestamp: float = field(default_factory=time.time)
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class PredictionResponse:
    """Antwort des Endpunkts auf eine Anfrage."""
    request_id: str
    prediction: int
    confidence: float
    probabilities: List[float]
    latency_ms: float
    model_version: str
    timestamp: float = field(default_factory=time.time)

    def to_json(self) -> str:
        return json.dumps({
            'request_id':   self.request_id,
            'prediction':   self.prediction,
            'confidence':   round(self.confidence, 4),
            'latency_ms':   round(self.latency_ms, 2),
            'model_version': self.model_version,
            'timestamp':    self.timestamp
        }, indent=2)


class DeploymentEndpoint:
    """
    Simulierter REST-Endpunkt für Modell-Inferenz.

    Implementiert:
      - Anfrage-Validierung
      - Latenz-Messung
      - Request-ID-Generierung (MD5-Hash)
      - Einfaches Rate-Limiting
      - Request-Log
    """

    MAX_REQUESTS_PER_SECOND = 100

    def __init__(self, model: tf.keras.Model, model_version: str = 'v1.0.0'):
        self.model         = model
        self.model_version = model_version
        self.logger        = logging.getLogger('DeploymentEndpoint')
        self._request_log: List[PredictionResponse] = []
        self._request_count = 0
        self._window_start  = time.time()
        self.logger.info(f"Endpunkt gestartet | Version: {model_version}")

    # ── Interne Hilfsmethoden ─────────────────────────────
    @staticmethod
    def _generate_request_id(payload: np.ndarray) -> str:
        """Eindeutige ID auf Basis von Payload-Hash und Zeit."""
        h = hashlib.md5(payload.tobytes()).hexdigest()[:8]
        t = str(time.time_ns())[-6:]
        return f"req-{h}-{t}"

    def _check_rate_limit(self) -> bool:
        """Einfaches Token-Bucket Rate-Limiting."""
        now = time.time()
        if now - self._window_start > 1.0:
            self._window_start  = now
            self._request_count = 0
        self._request_count += 1
        return self._request_count <= self.MAX_REQUESTS_PER_SECOND

    def _validate(self, payload: np.ndarray) -> bool:
        """Grundlegende Eingabevalidierung."""
        if payload is None or not isinstance(payload, np.ndarray):
            return False
        if payload.ndim not in (3, 4):   # (H,W,C) oder (N,H,W,C)
            return False
        return True

    # ── Öffentliche API ───────────────────────────────────
    def predict(self, payload: np.ndarray,
                metadata: Optional[Dict] = None) -> PredictionResponse:
        """Verarbeitet eine Einzelanfrage und gibt eine Antwort zurück."""
        if not self._check_rate_limit():
            raise RuntimeError("Rate-Limit überschritten.")
        if not self._validate(payload):
            raise ValueError("Ungültige Eingabedaten.")

        # Batch-Dimension ergänzen falls nötig
        if payload.ndim == 3:
            payload = payload[np.newaxis, ...]

        req_id = self._generate_request_id(payload)
        t0     = time.perf_counter()
        probs  = self.model.predict(payload, verbose=0)[0]
        latency_ms = (time.perf_counter() - t0) * 1000

        pred   = int(np.argmax(probs))
        conf   = float(probs[pred])

        response = PredictionResponse(
            request_id=req_id,
            prediction=pred,
            confidence=conf,
            probabilities=probs.tolist(),
            latency_ms=latency_ms,
            model_version=self.model_version
        )
        self._request_log.append(response)
        return response

    def batch_predict(self, payloads: np.ndarray) -> List[PredictionResponse]:
        """Verarbeitet mehrere Anfragen als Batch."""
        return [self.predict(p) for p in payloads]

    def get_stats(self) -> Dict[str, Any]:
        """Gibt Endpunkt-Statistiken zurück."""
        if not self._request_log:
            return {'total_requests': 0}
        latencies = [r.latency_ms for r in self._request_log]
        return {
            'total_requests':   len(self._request_log),
            'mean_latency_ms':  round(np.mean(latencies), 2),
            'p95_latency_ms':   round(np.percentile(latencies, 95), 2),
            'p99_latency_ms':   round(np.percentile(latencies, 99), 2),
            'model_version':    self.model_version
        }


# ============================================================
# 7. MODEL MONITOR
# ============================================================

class ModelMonitor:
    """
    Überwacht Modell-Performance und erkennt Datendrift.

    Implementiert:
      - Performance-Tracking über Zeitfenster
      - Konfidenz-Drift-Erkennung (Mittelwert-Shift)
      - Prediction-Distribution-Shift (Chi-Quadrat-ähnlich)
      - Alert-System bei Schwellenwertunterschreitungen
    """

    def __init__(self, window_size: int = 200,
                 alert_threshold_acc: float = 0.90,
                 alert_threshold_drift: float = 0.15):
        self.window_size            = window_size
        self.alert_threshold_acc    = alert_threshold_acc
        self.alert_threshold_drift  = alert_threshold_drift
        self.logger                 = logging.getLogger('ModelMonitor')

        # Ringpuffer für aktuelle Fenster-Metriken
        self._confidences:   List[float] = []
        self._correct:       List[bool]  = []
        self._pred_counts:   np.ndarray  = np.zeros(10, dtype=int)
        self._alerts:        List[Dict]  = []

        # Referenz-Baseline (wird beim ersten Aufruf gesetzt)
        self._baseline_conf:  Optional[float]     = None
        self._baseline_dist:  Optional[np.ndarray] = None

    def log_prediction(self, prediction: int, confidence: float,
                       true_label: Optional[int] = None):
        """Registriert eine Vorhersage im Monitor."""
        self._confidences.append(confidence)
        self._pred_counts[prediction] += 1

        if true_label is not None:
            self._correct.append(prediction == true_label)

        # Sliding window: älteste Einträge entfernen
        if len(self._confidences) > self.window_size:
            self._confidences.pop(0)
        if len(self._correct) > self.window_size:
            self._correct.pop(0)

    def set_baseline(self):
        """Setzt die aktuelle Verteilung als Referenz-Baseline."""
        if len(self._confidences) == 0:
            self.logger.warning("Keine Daten für Baseline vorhanden.")
            return
        self._baseline_conf = np.mean(self._confidences)
        total = self._pred_counts.sum()
        self._baseline_dist = (self._pred_counts / max(total, 1)).copy()
        self.logger.info(
            f"Baseline gesetzt: Konfidenz={self._baseline_conf:.4f}, "
            f"Dist={np.round(self._baseline_dist, 3)}"
        )

    def check_drift(self) -> Dict[str, Any]:
        """Prüft auf Datendrift im Vergleich zur Baseline."""
        report: Dict[str, Any] = {
            'timestamp':        datetime.datetime.now().isoformat(),
            'window_size':      len(self._confidences),
            'conf_drift':       None,
            'dist_drift':       None,
            'accuracy':         None,
            'alerts':           []
        }

        if len(self._confidences) < 10:
            return report

        curr_conf = np.mean(self._confidences)
        report['mean_confidence'] = round(curr_conf, 4)

        # Konfidenz-Drift
        if self._baseline_conf is not None:
            conf_drift = abs(curr_conf - self._baseline_conf)
            report['conf_drift'] = round(conf_drift, 4)
            if conf_drift > self.alert_threshold_drift:
                alert = {'type': 'CONFIDENCE_DRIFT',
                         'value': conf_drift,
                         'threshold': self.alert_threshold_drift}
                report['alerts'].append(alert)
                self._alerts.append(alert)
                self.logger.warning(f"ALERT: Konfidenz-Drift erkannt: {conf_drift:.4f}")

        # Vorhersage-Verteilungs-Drift (TVD – Total Variation Distance)
        if self._baseline_dist is not None:
            total = self._pred_counts.sum()
            curr_dist = self._pred_counts / max(total, 1)
            tvd = 0.5 * np.abs(curr_dist - self._baseline_dist).sum()
            report['dist_drift'] = round(float(tvd), 4)
            if tvd > self.alert_threshold_drift:
                alert = {'type': 'DISTRIBUTION_DRIFT', 'tvd': float(tvd)}
                report['alerts'].append(alert)
                self._alerts.append(alert)
                self.logger.warning(f"ALERT: Verteilungs-Drift erkannt: TVD={tvd:.4f}")

        # Accuracy-Alert
        if len(self._correct) >= 20:
            acc = np.mean(self._correct)
            report['accuracy'] = round(float(acc), 4)
            if acc < self.alert_threshold_acc:
                alert = {'type': 'LOW_ACCURACY', 'value': acc,
                         'threshold': self.alert_threshold_acc}
                report['alerts'].append(alert)
                self.logger.warning(f"ALERT: Niedrige Accuracy: {acc:.4f}")

        return report

    @property
    def all_alerts(self) -> List[Dict]:
        return self._alerts


# ============================================================
# 8. A/B TEST FRAMEWORK
# ============================================================

@dataclass
class ABTestConfig:
    """Konfiguration eines A/B-Experiments."""
    experiment_id: str
    model_a_name:  str
    model_b_name:  str
    traffic_split: float = 0.5    # Anteil für Modell A (0–1)
    min_samples:   int   = 100
    significance:  float = 0.05   # p-Wert-Schwelle


class ABTestFramework:
    """
    A/B-Test-Framework für zwei Modellversionen.

    Verteilt Traffic stochastisch anhand von traffic_split,
    sammelt Metriken pro Modell und führt einen einfachen
    z-Test für Proportionen durch (H₀: p_A = p_B).
    """

    def __init__(self, config: ABTestConfig,
                 model_a: tf.keras.Model,
                 model_b: tf.keras.Model):
        self.config  = config
        self.model_a = model_a
        self.model_b = model_b
        self.logger  = logging.getLogger('ABTestFramework')

        self._results: Dict[str, List] = defaultdict(list)
        self._results['A_correct']  = []
        self._results['B_correct']  = []
        self._results['A_latency']  = []
        self._results['B_latency']  = []
        self._results['A_conf']     = []
        self._results['B_conf']     = []

        self.logger.info(
            f"Experiment '{config.experiment_id}' gestartet | "
            f"Split: {config.traffic_split:.0%} A / "
            f"{1-config.traffic_split:.0%} B"
        )

    def route(self) -> str:
        """Gibt 'A' oder 'B' zurück basierend auf traffic_split."""
        return 'A' if np.random.random() < self.config.traffic_split else 'B'

    def record(self, payload: np.ndarray, true_label: int):
        """Verarbeitet eine Anfrage, zeichnet Ergebnis auf."""
        variant = self.route()
        model   = self.model_a if variant == 'A' else self.model_b
        key     = variant

        if payload.ndim == 3:
            payload = payload[np.newaxis, ...]

        t0    = time.perf_counter()
        probs = model.predict(payload, verbose=0)[0]
        lat   = (time.perf_counter() - t0) * 1000
        pred  = int(np.argmax(probs))
        conf  = float(probs[pred])

        self._results[f'{key}_correct'].append(pred == true_label)
        self._results[f'{key}_latency'].append(lat)
        self._results[f'{key}_conf'].append(conf)

    def analyze(self) -> Dict[str, Any]:
        """
        Führt z-Test für Proportionen durch und gibt Report zurück.

        H₀: Accuracy_A = Accuracy_B
        """
        n_a = len(self._results['A_correct'])
        n_b = len(self._results['B_correct'])

        report = {
            'experiment_id': self.config.experiment_id,
            'n_A': n_a, 'n_B': n_b,
            'sufficient_data': n_a >= self.config.min_samples and
                               n_b >= self.config.min_samples
        }

        if n_a == 0 or n_b == 0:
            report['status'] = 'Nicht genug Daten'
            return report

        acc_a = np.mean(self._results['A_correct'])
        acc_b = np.mean(self._results['B_correct'])
        report['acc_A'] = round(float(acc_a), 4)
        report['acc_B'] = round(float(acc_b), 4)
        report['mean_latency_A_ms'] = round(float(np.mean(self._results['A_latency'])), 2)
        report['mean_latency_B_ms'] = round(float(np.mean(self._results['B_latency'])), 2)
        report['mean_conf_A'] = round(float(np.mean(self._results['A_conf'])), 4)
        report['mean_conf_B'] = round(float(np.mean(self._results['B_conf'])), 4)

        # z-Test für Proportionen (wenn genug Daten)
        if report['sufficient_data']:
            p_pool = (acc_a * n_a + acc_b * n_b) / (n_a + n_b)
            se     = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
            if se > 0:
                z_stat  = (acc_a - acc_b) / se
                # Näherung für p-Wert (zweiseitig, Normalapproximation)
                p_value = 2 * (1 - min(0.9999,
                               0.5 * (1 + float(tf.math.erf(
                                   abs(z_stat) / np.sqrt(2))))))
                report['z_statistic'] = round(float(z_stat), 4)
                report['p_value']     = round(float(p_value), 4)
                report['significant'] = p_value < self.config.significance

                winner = None
                if report['significant']:
                    winner = self.config.model_a_name if acc_a > acc_b \
                             else self.config.model_b_name
                report['winner'] = winner

        return report


# ============================================================
# 9. ORCHESTRATOR – ALLES ZUSAMMENFÜHREN
# ============================================================

class AISystemOrchestrator:
    """
    Orchestriert alle Komponenten des vollständigen KI-Systems.

    Ablauf:
      1. Datenpipeline aufbauen
      2. CNN-Modell trainieren
      3. Leichtes Alternativmodell trainieren (für Ensemble/A/B)
      4. Ensemble erstellen und evaluieren
      5. Unsicherheitsschätzung demonstrieren
      6. Deployment-Endpunkt simulieren
      7. Monitor kalibrieren und Drift prüfen
      8. A/B-Test durchführen
      9. Bericht und Visualisierungen erstellen
    """

    def __init__(self):
        self.logger = logging.getLogger('Orchestrator')
        self.cfg    = DataConfig(batch_size=128, augment=True)

        # Komponentenreferenzen
        self.pipeline:   Optional[DataPipeline]         = None
        self.cnn:        Optional[CNNFeatureExtractor]   = None
        self.alt_model:  Optional[tf.keras.Model]        = None
        self.ensemble:   Optional[EnsemblePredictor]     = None
        self.estimator:  Optional[UncertaintyEstimator]  = None
        self.endpoint:   Optional[DeploymentEndpoint]    = None
        self.monitor:    Optional[ModelMonitor]          = None
        self.ab_test:    Optional[ABTestFramework]       = None

        # Ergebnis-Speicher
        self.history_cnn: Any = None
        self.history_alt: Any = None
        self.report:      Dict = {}

    # ── Schritt 1: Daten ──────────────────────────────────
    def setup_data(self):
        self.logger.info("=" * 50)
        self.logger.info("SCHRITT 1: Datenpipeline")
        self.pipeline = DataPipeline(self.cfg).load().build()

    # ── Schritt 2: CNN trainieren ─────────────────────────
    def train_cnn(self, epochs: int = 5):
        self.logger.info("=" * 50)
        self.logger.info("SCHRITT 2: CNN-Feature-Extraktor trainieren")
        train_ds, val_ds, _ = self.pipeline.get_splits()

        self.cnn = CNNFeatureExtractor(embedding_dim=128, dropout_rate=0.3)
        self.cnn.compile(
            optimizer=tf.keras.optimizers.Adam(1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        self.history_cnn = self.cnn.fit(
            train_ds, validation_data=val_ds,
            epochs=epochs,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    patience=3, restore_best_weights=True, monitor='val_accuracy')
            ],
            verbose=1
        )
        val_acc = max(self.history_cnn.history['val_accuracy'])
        self.logger.info(f"CNN – beste Val-Accuracy: {val_acc:.4f}")

    # ── Schritt 3: Alternativmodell trainieren ─────────────
    def train_alt_model(self, epochs: int = 4):
        self.logger.info("=" * 50)
        self.logger.info("SCHRITT 3: Alternatives Modell (leichtes CNN)")
        train_ds, val_ds, _ = self.pipeline.get_splits()

        inp = tf.keras.Input(shape=(28, 28, 1))
        x   = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
        x   = tf.keras.layers.MaxPooling2D(2)(x)
        x   = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
        x   = tf.keras.layers.GlobalAveragePooling2D()(x)
        x   = tf.keras.layers.Dense(64, activation='relu')(x)
        x   = tf.keras.layers.Dropout(0.3)(x)
        out = tf.keras.layers.Dense(10, activation='softmax')(x)

        self.alt_model = tf.keras.Model(inp, out, name='LightCNN')
        self.alt_model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        self.history_alt = self.alt_model.fit(
            train_ds, validation_data=val_ds,
            epochs=epochs, verbose=1
        )
        val_acc = max(self.history_alt.history['val_accuracy'])
        self.logger.info(f"LightCNN – beste Val-Accuracy: {val_acc:.4f}")

    # ── Schritt 4: Ensemble evaluieren ────────────────────
    def evaluate_ensemble(self, n_samples: int = 1000):
        self.logger.info("=" * 50)
        self.logger.info("SCHRITT 4: Ensemble-Evaluation")
        x_test, y_test = self.pipeline.raw_test
        x_sub, y_sub   = x_test[:n_samples], y_test[:n_samples]

        # Gewichte proportional zu Val-Accuracy
        acc_cnn = max(self.history_cnn.history['val_accuracy'])
        acc_alt = max(self.history_alt.history['val_accuracy'])
        self.ensemble = EnsemblePredictor.from_accuracies(
            models=[self.cnn, self.alt_model],
            accuracies=[acc_cnn, acc_alt]
        )
        metrics = self.ensemble.evaluate(x_sub, y_sub)
        self.report['ensemble_metrics'] = metrics

    # ── Schritt 5: Unsicherheitsschätzung ─────────────────
    def estimate_uncertainty(self, n_samples: int = 200):
        self.logger.info("=" * 50)
        self.logger.info("SCHRITT 5: MC-Dropout Unsicherheitsschätzung")
        x_test, _ = self.pipeline.raw_test
        x_sub     = x_test[:n_samples]

        self.estimator = UncertaintyEstimator(self.cnn, n_samples=20)
        mean_probs, unc, entropy = self.estimator.predict_with_uncertainty(x_sub)

        self.report['uncertainty'] = {
            'mean_epistemic_unc': round(float(unc.mean()), 6),
            'mean_pred_entropy':  round(float(entropy.mean()), 4),
            'high_unc_fraction':  round(float((unc > 0.01).mean()), 4)
        }
        reliable_idx, _ = self.estimator.get_confident_predictions(x_sub)
        self.report['uncertainty']['reliable_fraction'] = \
            round(len(reliable_idx) / n_samples, 4)
        self.logger.info(f"Unsicherheits-Report: {self.report['uncertainty']}")

    # ── Schritt 6: Deployment-Endpunkt ────────────────────
    def simulate_endpoint(self, n_requests: int = 100):
        self.logger.info("=" * 50)
        self.logger.info(f"SCHRITT 6: Endpoint-Simulation ({n_requests} Anfragen)")
        x_test, _ = self.pipeline.raw_test

        self.endpoint = DeploymentEndpoint(self.cnn, model_version='v2.0.0')
        for i in range(n_requests):
            self.endpoint.predict(x_test[i])

        stats = self.endpoint.get_stats()
        self.report['endpoint_stats'] = stats
        self.logger.info(f"Endpunkt-Stats: {stats}")

    # ── Schritt 7: Monitor kalibrieren + Drift prüfen ─────
    def run_monitor(self, n_log: int = 300):
        self.logger.info("=" * 50)
        self.logger.info("SCHRITT 7: Modell-Monitor")
        x_test, y_test = self.pipeline.raw_test

        self.monitor = ModelMonitor(window_size=150)

        # Baseline mit ersten 150 Samples aufbauen
        probs_base = self.cnn.predict(x_test[:150], verbose=0)
        for i in range(150):
            pred = int(np.argmax(probs_base[i]))
            conf = float(probs_base[i][pred])
            self.monitor.log_prediction(pred, conf, int(y_test[i]))
        self.monitor.set_baseline()

        # Weiteren Datenstrom einloggen (simulierter Drift)
        probs_new = self.cnn.predict(x_test[150:150 + n_log], verbose=0)
        for i in range(n_log):
            pred = int(np.argmax(probs_new[i]))
            conf = float(probs_new[i][pred])
            self.monitor.log_prediction(pred, conf, int(y_test[150 + i]))

        drift_report = self.monitor.check_drift()
        self.report['drift'] = drift_report
        self.logger.info(f"Drift-Report: {drift_report}")

    # ── Schritt 8: A/B-Test ───────────────────────────────
    def run_ab_test(self, n_requests: int = 300):
        self.logger.info("=" * 50)
        self.logger.info(f"SCHRITT 8: A/B-Test ({n_requests} Anfragen)")
        x_test, y_test = self.pipeline.raw_test

        ab_cfg = ABTestConfig(
            experiment_id='exp_cnn_vs_lightcnn_v1',
            model_a_name='CNN_FeatureExtractor',
            model_b_name='LightCNN',
            traffic_split=0.5,
            min_samples=100
        )
        self.ab_test = ABTestFramework(ab_cfg, self.cnn, self.alt_model)

        for i in range(n_requests):
            self.ab_test.record(x_test[i], int(y_test[i]))

        ab_report = self.ab_test.analyze()
        self.report['ab_test'] = ab_report
        self.logger.info(f"A/B-Report: {ab_report}")

    # ── Schritt 9: Visualisierungen ───────────────────────
    def visualize(self):
        self.logger.info("=" * 50)
        self.logger.info("SCHRITT 9: Visualisierungen erstellen")

        fig = plt.figure(figsize=(18, 12))
        fig.suptitle("Vollständiges KI-System – Evaluations-Dashboard",
                     fontsize=16, fontweight='bold')
        gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

        # ── A) Training-Kurven CNN ────────────────────────
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.plot(self.history_cnn.history['accuracy'],     label='Train')
        ax1.plot(self.history_cnn.history['val_accuracy'], label='Val', ls='--')
        ax1.set_title('CNN – Accuracy')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
        ax1.legend(); ax1.grid(alpha=0.3)

        ax2 = fig.add_subplot(gs[0, 1])
        ax2.plot(self.history_cnn.history['loss'],     label='Train')
        ax2.plot(self.history_cnn.history['val_loss'], label='Val', ls='--')
        ax2.set_title('CNN – Loss')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
        ax2.legend(); ax2.grid(alpha=0.3)

        # ── B) Ensemble Metriken ──────────────────────────
        ax3 = fig.add_subplot(gs[0, 2])
        em  = self.report.get('ensemble_metrics', {})
        metrics_names  = list(em.keys())
        metrics_values = list(em.values())
        colors = ['steelblue', 'seagreen', 'darkorange'][:len(metrics_names)]
        bars = ax3.bar(metrics_names, metrics_values, color=colors)
        ax3.set_ylim(0, 1.05)
        ax3.set_title('Ensemble-Metriken')
        ax3.set_ylabel('Wert')
        for bar, val in zip(bars, metrics_values):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.3f}', ha='center', fontsize=9)
        ax3.grid(axis='y', alpha=0.3)

        # ── C) Unsicherheits-Verteilung ───────────────────
        ax4 = fig.add_subplot(gs[1, 0])
        x_test, _ = self.pipeline.raw_test
        _, unc, entropy = self.estimator.predict_with_uncertainty(x_test[:200])
        ax4.hist(unc, bins=30, color='tomato', edgecolor='white', alpha=0.8)
        ax4.set_title('Epistemic Uncertainty')
        ax4.set_xlabel('Unsicherheit'); ax4.set_ylabel('Häufigkeit')
        ax4.axvline(unc.mean(), color='darkred', ls='--',
                    label=f'μ={unc.mean():.5f}')
        ax4.legend(); ax4.grid(alpha=0.3)

        ax5 = fig.add_subplot(gs[1, 1])
        ax5.hist(entropy, bins=30, color='mediumpurple', edgecolor='white', alpha=0.8)
        ax5.set_title('Prädiktive Entropie')
        ax5.set_xlabel('Entropie'); ax5.set_ylabel('Häufigkeit')
        ax5.axvline(entropy.mean(), color='indigo', ls='--',
                    label=f'μ={entropy.mean():.3f}')
        ax5.legend(); ax5.grid(alpha=0.3)

        # ── D) Endpoint Latenz-Verteilung ─────────────────
        ax6 = fig.add_subplot(gs[1, 2])
        latencies = [r.latency_ms for r in self.endpoint._request_log]
        ax6.hist(latencies, bins=25, color='teal', edgecolor='white', alpha=0.8)
        ax6.set_title('Endpoint-Latenzen')
        ax6.set_xlabel('Latenz (ms)'); ax6.set_ylabel('Anzahl')
        stats = self.endpoint.get_stats()
        ax6.axvline(stats['mean_latency_ms'], color='darkgreen', ls='--',
                    label=f"p50={stats['mean_latency_ms']:.1f}ms")
        ax6.axvline(stats['p95_latency_ms'], color='red', ls=':',
                    label=f"p95={stats['p95_latency_ms']:.1f}ms")
        ax6.legend(fontsize=8); ax6.grid(alpha=0.3)

        # ── E) A/B-Test Ergebnisse ─────────────────────────
        ax7 = fig.add_subplot(gs[2, 0])
        ab  = self.report.get('ab_test', {})
        if 'acc_A' in ab and 'acc_B' in ab:
            models_ab = [ab.get('experiment_id', 'Exp').split('_')[1].upper(),
                         ab.get('experiment_id', 'Exp').split('_')[3].upper()]
            accs      = [ab['acc_A'], ab['acc_B']]
            bar_cols  = ['steelblue', 'coral']
            b = ax7.bar(models_ab, accs, color=bar_cols, edgecolor='white')
            ax7.set_ylim(min(accs) - 0.02, min(max(accs) + 0.05, 1.0))
            ax7.set_title('A/B-Test: Accuracy')
            ax7.set_ylabel('Accuracy')
            for bar, val in zip(b, accs):
                ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                         f'{val:.4f}', ha='center', fontsize=9)
            sig_txt = "Signifikant" if ab.get('significant') else "Nicht signifikant"
            ax7.set_xlabel(f"p={ab.get('p_value', '?')} ({sig_txt})", fontsize=9)
        ax7.grid(axis='y', alpha=0.3)

        ax8 = fig.add_subplot(gs[2, 1])
        if 'mean_latency_A_ms' in ab and 'mean_latency_B_ms' in ab:
            models_ab2 = ['Modell A (CNN)', 'Modell B (Light)']
            lats       = [ab['mean_latency_A_ms'], ab['mean_latency_B_ms']]
            ax8.bar(models_ab2, lats, color=['steelblue', 'coral'],
                    edgecolor='white')
            ax8.set_title('A/B-Test: Ø Latenz')
            ax8.set_ylabel('ms')
            ax8.grid(axis='y', alpha=0.3)

        # ── F) Gesamtzusammenfassung ───────────────────────
        ax9 = fig.add_subplot(gs[2, 2])
        ax9.axis('off')
        summary_lines = [
            "SYSTEM-ZUSAMMENFASSUNG",
            "─" * 32,
        ]
        if 'ensemble_metrics' in self.report:
            em = self.report['ensemble_metrics']
            summary_lines += [
                f"Ensemble Accuracy:   {em.get('accuracy', 0):.4f}",
                f"Top-3 Accuracy:      {em.get('top3_accuracy', 0):.4f}",
                f"Mittl. Konfidenz:    {em.get('mean_confidence', 0):.4f}",
            ]
        if 'uncertainty' in self.report:
            u = self.report['uncertainty']
            summary_lines += [
                f"Ø Unsicherheit:      {u.get('mean_epistemic_unc', 0):.6f}",
                f"Zuverlässige Pred:   {u.get('reliable_fraction', 0):.2%}",
            ]
        if 'endpoint_stats' in self.report:
            es = self.report['endpoint_stats']
            summary_lines += [
                f"Endpoint p95 Latenz: {es.get('p95_latency_ms', 0):.1f} ms",
                f"Modell-Version:      {es.get('model_version', '?')}",
            ]
        if 'ab_test' in self.report:
            ab2 = self.report['ab_test']
            winner = ab2.get('winner') or 'kein Sieger'
            summary_lines.append(f"A/B-Gewinner:        {winner}")

        ax9.text(0.05, 0.95, '\n'.join(summary_lines),
                 transform=ax9.transAxes, va='top', ha='left',
                 fontsize=9, fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow',
                           edgecolor='goldenrod', alpha=0.8))

        plt.savefig('E15_3_ki_system_dashboard.png', dpi=100, bbox_inches='tight')
        plt.show()
        self.logger.info("Dashboard gespeichert: E15_3_ki_system_dashboard.png")

    # ── Hauptablauf ────────────────────────────────────────
    def run(self):
        """Führt alle 9 Schritte des vollständigen KI-Systems aus."""
        self.logger.info("╔══════════════════════════════════════════════════╗")
        self.logger.info("║  VOLLSTÄNDIGES KI-SYSTEM – ABSCHLUSSPROJEKT      ║")
        self.logger.info("╚══════════════════════════════════════════════════╝")
        start = time.time()

        self.setup_data()
        self.train_cnn(epochs=5)
        self.train_alt_model(epochs=4)
        self.evaluate_ensemble(n_samples=1000)
        self.estimate_uncertainty(n_samples=200)
        self.simulate_endpoint(n_requests=100)
        self.run_monitor(n_log=300)
        self.run_ab_test(n_requests=300)
        self.visualize()

        elapsed = time.time() - start
        self.logger.info("=" * 50)
        self.logger.info(f"Gesamtlaufzeit: {elapsed:.1f} s")

        # Finaler Report
        self.logger.info("── FINALER REPORT ──────────────────────────────")
        self.logger.info(json.dumps(
            {k: v for k, v in self.report.items()
             if k not in ('drift',)},  # Drift-Report ggf. groß
            indent=2
        ))

        return self.report


# ============================================================
# HAUPT-EINSTIEGSPUNKT
# ============================================================

if __name__ == '__main__':
    logger.info("Starte vollständiges KI-System …")
    system = AISystemOrchestrator()
    final_report = system.run()

    # ── Abschlussmeldung ─────────────────────────────────────
    logger.info("")
    logger.info("╔══════════════════════════════════════════════════╗")
    logger.info("║   ABSCHLUSSPROJEKT ERFOLGREICH ABGESCHLOSSEN     ║")
    logger.info("╠══════════════════════════════════════════════════╣")

    em = final_report.get('ensemble_metrics', {})
    logger.info(f"║  Ensemble Accuracy:  {em.get('accuracy', 0):.4f}                       ║")
    logger.info(f"║  Top-3 Accuracy:     {em.get('top3_accuracy', 0):.4f}                       ║")

    es = final_report.get('endpoint_stats', {})
    logger.info(f"║  Endpoint-Anfragen:  {es.get('total_requests', 0):<5}                        ║")

    ab = final_report.get('ab_test', {})
    logger.info(f"║  A/B-Test Gewinner:  {str(ab.get('winner', 'N/A')):<30}║")
    logger.info("╚══════════════════════════════════════════════════╝")
    logger.info("")
    logger.info("Alle Komponenten demonstriert:")
    logger.info("  ✔  DataPipeline         (Augmentierung, Caching)")
    logger.info("  ✔  CNNFeatureExtractor  (BatchNorm, Dropout)")
    logger.info("  ✔  LSTMTemporalModel    (Bidirektional)")
    logger.info("  ✔  EnsemblePredictor    (Accuracy-gewichtet)")
    logger.info("  ✔  UncertaintyEstimator (MC-Dropout)")
    logger.info("  ✔  DeploymentEndpoint   (Latenz, Rate-Limit)")
    logger.info("  ✔  ModelMonitor         (Drift-Erkennung)")
    logger.info("  ✔  ABTestFramework      (z-Test für Proportionen)")
    logger.info("")
    logger.info("Ausgabedatei: E15_3_ki_system_dashboard.png")
